In [1]:
import os
import glob
import torch
from collections import Counter
from sklearn.model_selection import train_test_split

# Point to your specific folder
data_dir = "./FSM_dataset"  
file_paths = sorted(glob.glob(os.path.join(data_dir, "*.pt")))

# Read labels
labels = []
for path in file_paths:
    data_dict = torch.load(path, map_location="cpu")
    # Handle both tensor labels and raw numbers safely
    label = data_dict["y"].item() if isinstance(data_dict["y"], torch.Tensor) else data_dict["y"][0]
    labels.append(label)

# Quick check on your 6 classes distribution
print("Total files found:", len(file_paths))
print("Class distribution across your 2 classes:", Counter(labels))

Total files found: 740
Class distribution across your 2 classes: Counter({0: 370, 1: 370})


In [2]:
# Stage 1: Split into Train (80%) and a temporary Rest set (20%)
train_paths, rest_paths, train_labels, rest_labels = train_test_split(
    file_paths, 
    labels, 
    test_size=0.20, 
    random_state=42, 
    stratify=labels # Enforces equal class representation
)

# Stage 2: Split the Rest set equally into Validation (10%) and Test (10%)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    rest_paths, 
    rest_labels, 
    test_size=0.50, 
    random_state=42, 
    stratify=rest_labels # Enforces equal class representation here too
)

print(f"\n--- Split Results ---")
print(f"Train set : {len(train_paths)} files")
print(f"Val set   : {len(val_paths)} files")
print(f"Test set  : {len(test_paths)} files")


--- Split Results ---
Train set : 592 files
Val set   : 74 files
Test set  : 74 files


In [3]:
print("\nTrain class distribution:", Counter(train_labels))
print("Val class distribution:  ", Counter(val_labels))
print("Test class distribution: ", Counter(test_labels))


Train class distribution: Counter({1: 296, 0: 296})
Val class distribution:   Counter({1: 37, 0: 37})
Test class distribution:  Counter({1: 37, 0: 37})


In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}\n")

Using device: cuda



In [5]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool

# ==========================================
# 1. DEFINE YOUR GNN ARCHITECTURE
# ==========================================
class FSM(nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes):
        super().__init__()
        self.gin1 = GINConv(nn.Sequential(
            nn.Linear(num_features, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        ))
        self.gin2 = GINConv(nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        ))
        self.gin3 = GINConv(nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        ))
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch,num_graphs=None):
        x = F.relu(self.gin1(x, edge_index))
        x = F.relu(self.gin2(x, edge_index))
        x = F.relu(self.gin3(x, edge_index))
        x = global_mean_pool(x, batch,size=num_graphs)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.fc2(x)

# ==========================================
# 2. STRATIFIED SPLIT (THE 300 FILES)
# ==========================================
data_dir = "./FSM_dataset"  # Your folder name
file_paths = sorted(glob.glob(os.path.join(data_dir, "*.pt")))

# Read labels from disk
labels = []
for path in file_paths:
    data_dict = torch.load(path, map_location="cpu")
    label = data_dict["y"].item() if isinstance(data_dict["y"], torch.Tensor) else data_dict["y"][0]
    labels.append(label)

# 80/10/10 Split Strategy
train_paths, rest_paths, train_labels, rest_labels = train_test_split(
    file_paths, labels, test_size=0.20, random_state=42, stratify=labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    rest_paths, rest_labels, test_size=0.50, random_state=42, stratify=rest_labels
)

print(f"Dataset Split Details:")
print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}\n")

# ==========================================
# 3. PYG CUSTOM DATASET & LOADERS
# ==========================================
class StratifiedGraphDataset(Dataset):
    def __init__(self, paths):
        super().__init__()
        self.paths = paths

    def len(self):
        return len(self.paths)

    def get(self, idx):
        data_dict = torch.load(self.paths[idx], map_location="cpu")
        return Data(x=data_dict["x"], edge_index=data_dict["edge_index"], y=data_dict["y"])

# Connect paths to dataset instances
train_dataset = StratifiedGraphDataset(train_paths)
val_dataset = StratifiedGraphDataset(val_paths)
test_dataset = StratifiedGraphDataset(test_paths)

# Connect datasets to data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==========================================
# 4. HARDWARE DEVICE DETECTION (LAPTOP GPU)
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")  # Dynamic support for Apple Silicon Macs
else:
    device = torch.device("cpu")

print(f"Running execution pipeline on device: {device}\n")

# ==========================================
# 5. INITIALIZE MODEL & TRAINING LOOP
# ==========================================
# Look at the very first training sample to grab its feature count dynamically
NUM_FEATURES = train_dataset[0].num_features 
HIDDEN_DIM = 64
NUM_CLASSES = 2

model = FSM(num_features=NUM_FEATURES, hidden_dim=HIDDEN_DIM, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Simple Training Loop
for epoch in range(1,2001):  # Run for 20 epochs as a test run
    model.train()
    total_loss = 0
    correct = 0
    
    for batch in train_loader:
        batch = batch.to(device)  # Moves the graph structures to your GPU
        
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=-1)
        correct += int((pred == batch.y).sum())
        
    train_loss = total_loss / len(train_dataset)
    train_acc = correct / len(train_dataset)
    
    # Quick validation pass every epoch to check your model's progress
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            pred = out.argmax(dim=-1)
            val_correct += int((pred == batch.y).sum())
    val_acc = val_correct / len(val_dataset)
            
    print(f"Epoch {epoch:02d} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}")
    best_val_acc = 0.0  # Track the highest accuracy seen so far

for epoch in range(1, 201):
    # ... [your train_loader loop here] ...
    
    # ... [your val_loader loop here] ...
    
    print(f"Epoch {epoch:03d} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}")
    
    # Save the model if validation accuracy improves
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "FSM_classifier.pth")
        print(f" => Saved new best model with Val Acc: {val_acc:.2%}")

C:\Users\vaidy\torch_env\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\vaidy\torch_env\Lib\site-packages\torch_scatter\_version_cuda.pyd
  import torch_geometric.typing
C:\Users\vaidy\torch_env\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\vaidy\torch_env\Lib\site-packages\torch_sparse\_version_cuda.pyd
  import torch_geometric.typing
C:\Users\vaidy\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset Split Details:
Train: 592 | Val: 74 | Test: 74

Running execution pipeline on device: cuda

Epoch 01 | Loss: 0.6938 | Train Acc: 50.51% | Val Acc: 50.00%
Epoch 02 | Loss: 0.6949 | Train Acc: 51.01% | Val Acc: 50.00%
Epoch 03 | Loss: 0.6921 | Train Acc: 54.05% | Val Acc: 50.00%
Epoch 04 | Loss: 0.6948 | Train Acc: 49.16% | Val Acc: 50.00%
Epoch 05 | Loss: 0.6932 | Train Acc: 50.68% | Val Acc: 50.00%
Epoch 06 | Loss: 0.6924 | Train Acc: 52.20% | Val Acc: 50.00%
Epoch 07 | Loss: 0.6930 | Train Acc: 50.68% | Val Acc: 50.00%
Epoch 08 | Loss: 0.6951 | Train Acc: 51.35% | Val Acc: 50.00%
Epoch 09 | Loss: 0.6912 | Train Acc: 52.70% | Val Acc: 48.65%
Epoch 10 | Loss: 0.6928 | Train Acc: 50.34% | Val Acc: 62.16%
Epoch 11 | Loss: 0.6941 | Train Acc: 50.00% | Val Acc: 50.00%
Epoch 12 | Loss: 0.6936 | Train Acc: 48.99% | Val Acc: 50.00%
Epoch 13 | Loss: 0.6960 | Train Acc: 47.30% | Val Acc: 50.00%
Epoch 14 | Loss: 0.6943 | Train Acc: 48.99% | Val Acc: 50.00%
Epoch 15 | Loss: 0.6937 | Train 

KeyboardInterrupt: 

In [6]:
import torch

# Define the file name where you want to store the brain of your network
MODEL_FILE = "FSM_classifier.pth"

# Save the weights
torch.save(model.state_dict(), MODEL_FILE)
print(f"✅ Model weights saved securely to {MODEL_FILE}")

✅ Model weights saved securely to FSM_classifier.pth


In [7]:
# 1. Load your best saved model weights (if you used the checkpoint saver)
# model.load_state_dict(torch.load("best_circuit_gin.pth"))

# 2. Run the final evaluation on the completely unseen Test Set
model.eval()
test_correct = 0

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        pred = out.argmax(dim=-1)
        test_correct += int((pred == batch.y).sum())

test_acc = test_correct / len(test_dataset)
print(f"🚀 Final Unseen Test Accuracy: {test_acc:.2%}")

🚀 Final Unseen Test Accuracy: 100.00%


In [8]:
import os

model.eval()
misclassified_circuits = []
global_test_idx = 0  # Tracks the absolute position in test_paths

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        
        # Forward pass (using the empty-graph fix from earlier)
        out = model(batch.x, batch.edge_index, batch.batch, num_graphs=batch.num_graphs)
        preds = out.argmax(dim=-1)
        
        # Loop through each individual graph inside the current batch
        for local_idx in range(batch.num_graphs):
            actual_label = batch.y[local_idx].item()
            predicted_label = preds[local_idx].item()
            
            # If the model guessed wrong, capture the details
            if predicted_label != actual_label:
                exact_file_path = test_paths[global_test_idx]
                misclassified_circuits.append({
                    "filename": os.path.basename(exact_file_path),
                    "true_class": actual_label,
                    "pred_class": predicted_label
                })
            
            # Move to the next file path pointer
            global_test_idx += 1

# ==========================================
# PRINT THE CONFUSION REPORT
# ==========================================
print(f"🔍 Found {len(misclassified_circuits)} Misclassified Circuits:\n")
print(f"{'Filename':<25} | {'True Class':<10} | {'Predicted Class':<15}")
print("-" * 60)

for circuit in misclassified_circuits:
    print(f"{circuit['filename']:<25} | {circuit['true_class']:<10} | {circuit['pred_class']:<15}")

🔍 Found 0 Misclassified Circuits:

Filename                  | True Class | Predicted Class
------------------------------------------------------------


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv, global_mean_pool

# 1. Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Re-create the structural blueprint (Must match your original architecture exactly)
# Replace 'NUM_FEATURES' with your specific node feature count (e.g., 10, 56, etc.)
NUM_FEATURES = train_dataset[0].num_features 

loaded_model = FSM(num_features=NUM_FEATURES, hidden_dim=64, num_classes=2)

# 3. Inject your saved weights into this blueprint
loaded_model.load_state_dict(torch.load("FSM_classifier.pth", map_location=device))
loaded_model = loaded_model.to(device)

# 4. CRITICAL: Switch model to evaluation mode (Turns off Dropout)
loaded_model.eval()

print("🚀 Model successfully loaded and frozen for individual testing!")

🚀 Model successfully loaded and frozen for individual testing!


In [11]:
# Choose the specific file you want to test
single_file_path = "./FSM_dataset/source_369_mealy.pt" 

# 1. Load the dictionary from disk
data_dict = torch.load(single_file_path, map_location="cpu")

# 2. Extract the core Graph components
x = data_dict["x"].to(device)                   # Node features
edge_index = data_dict["edge_index"].to(device) # Node connections

# 3. THE TRICK: Create a dummy batch vector
# Every single node needs to point to graph index '0'
num_nodes = x.size(0)
batch_vector = torch.zeros(num_nodes, dtype=torch.long, device=device)

print(f"Loaded '{single_file_path}' containing {num_nodes} circuit components.")

Loaded './FSM_dataset/source_369_mealy.pt' containing 29 circuit components.


In [12]:
with torch.no_grad(): # Tells PyTorch not to calculate gradients (saves VRAM)
    
    # Run the graph through your model
    raw_output = loaded_model(x, edge_index, batch_vector, num_graphs=1)
    
    # Convert raw outputs to probabilities (0.0 to 1.0)
    probabilities = F.softmax(raw_output, dim=-1)
    
    # Pick the class index with the absolute highest probability
    predicted_class = raw_output.argmax(dim=-1).item()
    confidence = probabilities[0][predicted_class].item()

# ==========================================
# SHOW THE RESULTS
# ==========================================
print("\n--- Model Prediction Report ---")
print(f"🎯 Predicted Circuit Class: Class {predicted_class}")
print(f"🔥 Model Confidence Score : {confidence:.2%}")

# Optional: Print the ground truth if the file contains a label 'y'
if "y" in data_dict:
    true_label = data_dict["y"].item() if isinstance(data_dict["y"], torch.Tensor) else data_dict["y"]
    print(f"📊 Actual Ground Truth    : Class {true_label}")
    
    if predicted_class == true_label:
        print("✅ Correct Prediction!")
    else:
        print("❌ Misclassified!")
print("--------------------------------")


--- Model Prediction Report ---
🎯 Predicted Circuit Class: Class 0
🔥 Model Confidence Score : 100.00%
📊 Actual Ground Truth    : Class 0
✅ Correct Prediction!
--------------------------------


In [13]:
for class_id, prob in enumerate(probabilities[0]):
    print(f"Class {class_id}: {prob.item():.2%}")

Class 0: 100.00%
Class 1: 0.00%
